<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_65_multimodal_vlm_whisper_tts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🎨 **Module 65 — Multimodal (Vision-Language + Whisper + TTS)** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 9 · Production Gaps


# Module 65 — Multimodal

> The course so far is **text-only**. But every modern AI product takes images, audio, and increasingly video. This module covers the *modalities you'll actually ship*: **Vision-Language Models (VLMs)** for "describe this screenshot," **Whisper** for speech-to-text, and the **TTS stack** for voice output. You'll see the architecture (it's the same transformer with an encoder swapped onto the front) and the production patterns that follow.

### What you'll cover
1. The multimodal landscape in one diagram
2. How a **Vision-Language Model** is built (CLIP → projector → LLM)
3. **CLIP** — encode an image into the *same vector space as text*
4. Run a real **VLM** (Qwen2-VL / LLaVA / Moondream) — image + question in, answer out
5. **Whisper** — production speech-to-text
6. **TTS** — Bark / Coqui XTTS / OpenAI TTS / ElevenLabs
7. End-to-end voice pipeline (audio → ASR → LLM → TTS → audio)
8. **Multimodal RAG** — how to index images
9. Costs, latency, and the hosted vs local trade-off
10. What's next: video, world models, MLLMs (Gemini, GPT-4o, Claude 3.5)


## 1 · The multimodal landscape

```
                ┌──────────── text-only LLMs (M16-M63) ─────────────┐
                │  GPT-2, Llama, Qwen — everything we built          │
                └────────────────────────────────────────────────────┘
                                          ▼  add modality encoders
                ┌────────────────────────────────────────────────────┐
                │   IMAGE   →  image encoder (CLIP/SigLIP)  →  ─┐    │
                │   AUDIO   →  audio encoder (Whisper)      →  ─┤    │
                │   VIDEO   →  video encoder (CLIP3D / V-JEPA)→ ─┼─→ │ LLM
                │   TEXT    →  tokenizer (M55)              →  ─┘    │
                └────────────────────────────────────────────────────┘
                                          ▼  decode
                ┌────────────────────────────────────────────────────┐
                │   TEXT     ←  LM head + sampler                    │
                │   AUDIO    ←  TTS (Bark, XTTS, …)                  │
                │   IMAGE    ←  diffusion model (SD, FLUX) — M21     │
                └────────────────────────────────────────────────────┘
```

The transformer body **doesn't change**. What changes is the **encoder on the front** and the **decoder on the back**. Most "multimodal LLMs" today are: take a pretrained LLM, add a small encoder + a 1-2 layer projection that maps encoder outputs into the LLM's embedding space, fine-tune jointly. That's it.

## 2 · VLM architecture in one picture

```
   image (224×224)
        │
        ▼
   ┌────────────────────┐
   │  ViT image encoder │  (CLIP / SigLIP / EVA — ~300M params)
   │  e.g. 256 patches  │
   └────────┬───────────┘
            │  256 patch embeddings
            ▼
   ┌──────────────────────┐
   │ MLP projector        │  (tiny — bridges encoder → LLM dims)
   │ d_image → d_model    │
   └──────────┬───────────┘
              │  256 'soft image tokens' in LLM-embedding space
              ▼
   ┌──────────────────────────────┐
   │ LLM decoder (Llama / Qwen)  │  ← concatenated with text tokens
   └──────────────────────────────┘
```

The image becomes a **sequence of "soft tokens"** in the LLM's own embedding space. They're prepended to (or interleaved with) the user's text tokens. From the LLM's perspective, **the image looks like extra tokens it can attend to.**

This is why most VLM training is cheap: the LLM body is already trained; you only fine-tune the **projector** and (optionally) a LoRA on the LLM.

## 3 · CLIP — the encoder that started it all

In [ ]:
!pip -q install transformers Pillow torch torchaudio

In [ ]:
# CLIP encodes images AND text into the SAME 512-d (or 768-d) vector space.
# Cosine similarity in that space tells you how well a caption matches an image.
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import urllib.request

# Grab a sample image
url = "https://images.cocodataset.org/val2017/000000039769.jpg"
img_path = "/content/cats.jpg"
urllib.request.urlretrieve(url, img_path)
img = Image.open(img_path)

model_id = "openai/clip-vit-base-patch32"
proc = CLIPProcessor.from_pretrained(model_id)
clip = CLIPModel.from_pretrained(model_id).eval()

captions = [
    "a photo of two cats sleeping on a couch",
    "a photo of a dog running on the beach",
    "a screenshot of a Python error traceback",
    "a city skyline at sunset",
]

inputs = proc(text=captions, images=img, return_tensors="pt", padding=True)
with torch.no_grad():
    out = clip(**inputs)

# logits_per_image is the image-vs-caption similarity matrix
probs = out.logits_per_image.softmax(dim=-1)[0]
for caption, p in zip(captions, probs):
    print(f"{p.item():.3f}  {caption}")

**That's CLIP.** One forward pass and you have an image embedding plus four text embeddings, all comparable in the same space. CLIP-style training is the foundation of every modern VLM:
- **Image search** — encode every photo once; encode the query text; nearest-neighbour.
- **Zero-shot classification** — embed `"a photo of a {label}"` for every label, pick the closest.
- **Safety filters** — embed banned-content captions, flag images close to them.
- **VLM front-end** — feed the image embeddings (the *patch* embeddings, not the final pooled vector) into an LLM.

## 4 · A real VLM — image-question-answering

For a true VLM that takes `(image, text question) → text answer` we use **Qwen2-VL** or **LLaVA** or **Moondream2** (very small).

In [ ]:
# We'll use Moondream2 — ~1.9B params, runs on free Colab CPU/T4.
# It's tiny but produces real captions and answers.
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "vikhyatk/moondream2"
moondream = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, trust_remote_code=True, revision="2024-08-26",
).eval()
tok = AutoTokenizer.from_pretrained(MODEL_ID, revision="2024-08-26")

# Encode the image once, then ask multiple questions
enc_image = moondream.encode_image(img)

for q in [
    "Describe this image in one sentence.",
    "How many cats are there?",
    "What surface are they on?",
]:
    a = moondream.answer_question(enc_image, q, tok)
    print(f"Q: {q}\nA: {a}\n")

**Notice the pattern.** Encode the image **once**, then run multiple questions against that cached encoding. That's the standard production shape — pay the image-encoder cost once per upload, run cheap text-only forwards per follow-up question.

### When to pick which VLM

| Model | Params | Cost | Quality |
|---|---|---|---|
| **Moondream2** | 1.9B | runs on a phone | great captions, fair OCR |
| **LLaVA-1.5/1.6** | 7-34B | one GPU | strong all-rounder |
| **Qwen2-VL** | 2-72B | one GPU+ | best open-source as of 2025 — strong OCR, video, multi-image |
| **Idefics-3 / Pixtral** | 8-12B | one GPU | strong on dense documents |
| **GPT-4o / Claude 3.5 Sonnet / Gemini 2.5** | hosted | $/image | frontier; complex reasoning, fine-grained OCR |


## 5 · Whisper — speech-to-text

In [ ]:
# Use a small Whisper variant so it runs on free Colab.
# whisper-tiny is ~75M params; whisper-large-v3 is ~1.5B (much better) on a GPU.
from transformers import pipeline

asr = pipeline(task="automatic-speech-recognition",
               model="openai/whisper-tiny.en",
               chunk_length_s=30)

# Sample audio: a public-domain 16-second clip
audio_url = "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/1.flac"
audio_path = "/content/sample.flac"
urllib.request.urlretrieve(audio_url, audio_path)

out = asr(audio_path)
print("transcript:", out["text"])

**Whisper specifics worth knowing.**
- **Multilingual + translation** — whisper-large-v3 transcribes 99 languages and can translate any of them *to English* in one pass.
- **Timestamps** — pass `return_timestamps=True` to get per-segment start/end times. Critical for subtitles, search, diarization.
- **Hallucinations** — Whisper invents text during long silences. Set `no_speech_threshold` higher; chunk audio with VAD (Silero VAD) first.
- **Speed**:
  - `faster-whisper` (CTranslate2 backend) is **4× faster** than HF Transformers with the same weights.
  - **`distil-whisper`** is 6× faster and 50% smaller with similar quality.
  - For real-time, **`whisper.cpp`** + `tiny`/`base` on an M1 hits 1× real-time on CPU.

## 6 · TTS — voice synthesis

TTS is messier than ASR. The trade-offs:

| Engine | Quality | Speed | Voice cloning | Latency |
|---|---|---|---|---|
| **OpenAI TTS** (tts-1, tts-1-hd) | very good | hosted, fast | no | ~700ms |
| **ElevenLabs** | best in class | hosted | yes (3-min sample) | ~400ms |
| **Coqui XTTS-v2** | very good | local GPU | yes (6-sec sample) | ~2s |
| **Bark** (Suno) | quirky / character voices | local GPU | clone via prompt | ~5s |
| **Piper** | decent | local CPU | no | ~100ms |
| **F5-TTS / Mimi / Kokoro-82M** (2024+) | top-tier local | local | yes | very fast |

For 99% of products the answer is **OpenAI TTS** or **ElevenLabs** until cost is an issue; then switch to **Piper** (CPU) or **F5-TTS / Kokoro** (GPU local). Voice cloning is the killer feature of the 2024+ generation.

In [ ]:
# Demo with a tiny Hugging Face TTS model that runs without a GPU.
from transformers import pipeline
import soundfile as sf

tts = pipeline("text-to-speech", model="microsoft/speecht5_tts")
# SpeechT5 needs a speaker embedding (xvector) — load one of the canonical ones
from datasets import load_dataset
embeddings_ds = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")
speaker_embedding = torch.tensor(embeddings_ds[7306]["xvector"]).unsqueeze(0)

result = tts("Multimodal AI is just text-only AI with a couple of encoders on the front.",
             forward_params={"speaker_embeddings": speaker_embedding})
sf.write("/content/out.wav", result["audio"], samplerate=result["sampling_rate"])
print("wrote /content/out.wav  shape:", result["audio"].shape, "  sr:", result["sampling_rate"])

You can play `/content/out.wav` in Colab with `IPython.display.Audio('/content/out.wav')`.

## 7 · End-to-end voice agent pipeline

```
   🎤 user audio
        │
        ▼  ASR (Whisper)              ← M65 §5
   "what's the weather in Paris?"
        │
        ▼  LLM (vLLM / OpenAI / local)  ← M44, M38
   "It's 21°C and sunny in Paris."
        │
        ▼  TTS (Piper / XTTS / ElevenLabs)  ← M65 §6
   🔊 spoken response
```

For real-time voice agents, latency stack matters more than absolute quality.

| Step | Target latency (real-time feel) |
|---|---|
| ASR (streaming) | first transcript chunk in **< 300 ms** |
| LLM | first token in **< 500 ms** (M44 vLLM prefix caching helps) |
| TTS (streaming) | first audio chunk in **< 200 ms** |
| **Total round-trip** | **< 1 s** |

That's why **Realtime APIs** (OpenAI Realtime, Gemini Live, ElevenLabs Conversational) bypass the three-step pipeline and use a single **speech-in/speech-out** model. They sacrifice some text-tool integration for sub-second back-and-forth.

## 8 · Multimodal RAG

When your corpus has images (slides, screenshots, PDFs with figures), text-only RAG misses them. Two patterns:

### Pattern A — caption-then-embed
1. For each image, run a VLM to produce a caption.
2. Embed the caption with `sentence-transformers`.
3. RAG normally (M30) against text + captions.

Simple, works, the model never *sees* the pixels — only your generated caption.

### Pattern B — visual embeddings (CLIP / SigLIP)
1. Embed each image with **CLIP** into a 512-d vector.
2. Embed each text chunk with the *same* CLIP into 512-d.
3. Store both in one vector DB (M42).
4. At query time, embed the query text with CLIP → find nearest neighbours regardless of modality.
5. Pass retrieved images **directly to a VLM** alongside the text context.

Visual embeddings preserve fidelity (the VLM sees the actual figure) but require a VLM to generate the answer.

### Pattern C — late interaction (**ColPali**, ColQwen)
The 2024 breakthrough: instead of pooling each image to one vector, store **per-patch** embeddings and compute MaxSim against query token embeddings at retrieval time. Best-in-class for PDF / slide retrieval.

## 9 · Costs and the hosted vs local trade-off

| Workload | Self-host? | Why |
|---|---|---|
| Light VLM usage (< 10k images/day) | hosted (GPT-4o, Claude 3.5, Gemini Flash) | ~$0.01/image; not worth GPU ops |
| Heavy VLM usage (millions of images) | local Qwen2-VL or LLaVA on A100 | break-even at ~50k images/day |
| ASR at any scale | local **distil-whisper** or **faster-whisper** | hosted ASR ~$0.006/min adds up fast |
| TTS — small consumer app | hosted (OpenAI / ElevenLabs) | per-character pricing, no GPU ops |
| TTS — large product | local **F5-TTS / Kokoro / Piper** | hosted TTS is ~$0.30/1k chars; local is free at scale |
| Voice cloning | ElevenLabs (cloud) or **XTTS-v2** (local) | legal compliance matters more than cost |

**Build the pipeline against hosted APIs first, then swap one component at a time when the cost actually bites.** Don't pre-optimise to local on day one.

## 10 · What's next — frontier multimodal

| Model / model family | What it adds |
|---|---|
| **GPT-4o** (2024) | speech-in / speech-out in one model; ~200 ms first audio |
| **Gemini 2.5 Pro / Flash** | million-token context; video understanding native |
| **Claude 3.5 Sonnet** (2024) | excellent OCR, complex chart reading, computer-use (M72) |
| **Qwen2-VL / Qwen2.5-VL** | best open-source VLM as of 2025; strong OCR + video |
| **V-JEPA / Sora / Veo / Movie Gen** | video generation; world models |
| **Voicebox / VALL-E / NaturalSpeech 3** | TTS with style + cloning from a few seconds |

### Where this fits in the course
- **M65** (this) — text-image-audio core building blocks.
- **M67** — alignment (DPO/RLHF) on multimodal models adds a "preference per (image, response)" wrinkle.
- **M72** — computer-use agents are VLMs that watch a screen and emit clicks.

## ✅ Recap
- A VLM is just a pretrained LLM with an image encoder + tiny projector on the front.
- **CLIP** encodes images and text into the same vector space — the workhorse of multimodal retrieval and zero-shot classification.
- **Whisper** is the production default for ASR; switch to `faster-whisper` / `distil-whisper` for speed.
- **TTS**: hosted (OpenAI / ElevenLabs) until cost bites; then `F5-TTS` / `Piper` / `XTTS` locally.
- Build pipelines against hosted APIs first. Swap one component to local when the bill justifies it.
- Multimodal RAG via caption-then-embed (simple), CLIP embeddings (preserves pixels), or **ColPali** late-interaction (best-in-class).

Next: **M66 — Distributed Training (DDP / FSDP / DeepSpeed)**.
